### SECTION04.일원 분산 분석

4개의 중학교에서 다른 교육 방법을 사용해 학생들에게 수학을 가르쳤다. 각 중학교의 학생들이 받은 교육 후의 수학 성적은 다음과 같다. 이 4개의 교육 방법에 따른 성적 차이가 통계적으로 유의미한지 5%의 유의수준에서 검정하시오.
##### 귀무가설 : 모든 중학교의 학생들의 수학 성적 평균은 동일하다.
##### 대립가설 : 최소 2개 중학교의 학생들의 수학 성적 평균은 다르다.(4개의 교육 방법 중 최소한 하나의 방법이 다른 방법들과 성적에서 차이가 있다.)

In [40]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [23]:
df = pd.read_csv('math.csv')
df.sample(5)

,groups,scores
23,group_C,84
0,group_A,85
15,group_B,78
8,group_A,88
22,group_C,85


1.각 그룹의 수학 성적에 대해 shapiro-wilk 검정을 통한 정규성을 확인하고 그 결과의 p-value를 구하시오.

In [24]:
cal = df['groups'].unique()
cal

for group in cal:
    cond = df['groups'] == group
    statistic, pvalue = stats.shapiro(df[cond]['scores'])
    
    print("====",group,"=====")
    print("statistic :", statistic)
    print("pvalue :", pvalue)

    if pvalue > 0.05:
        print("정규성 충족")
    else:
        print("정규성 불충족")

==== group_A =====
statistic : 0.9715896670696531
pvalue : 0.9051800443853569
정규성 충족
==== group_B =====
statistic : 0.9499422438060351
pvalue : 0.6678172590861611
정규성 충족
==== group_C =====
statistic : 0.9299424104842702
pvalue : 0.44732595113862045
정규성 충족
==== group_D =====
statistic : 0.9065684572704982
pvalue : 0.25824165549017347
정규성 충족


2.네 개 그룹의 수학 성적이 등분산성을 갖는지 확인하기 위해 levene 검정을 실시하고 p-value를 구하시오.

In [31]:
condA = df['groups'] == 'group_A'
condB = df['groups'] == 'group_B'
condC = df['groups'] == 'group_C'
condD = df['groups'] == 'group_D'

statistic, pvalue = stats.levene(df[condA]['scores'],df[condB]['scores'],df[condC]['scores'],df[condD]['scores'])
print("statistic :", statistic)
print("pvalue :", pvalue)

statistic : 1.757685352622062
pvalue : 0.17270284963232108


3.유의수준 0.05에서 귀무가설을 기준으로 검정의 결과를 채택/기각 중 선택해 입력하시오.

In [56]:
statistic,pvalue = stats.f_oneway(df[condA]['scores'],df[condB]['scores'],df[condC]['scores'],df[condD]['scores'])

print("===일원 분산 분석===")
print("statistic :", statistic)
print("pvalue :", pvalue)

if pvalue > 0.05:
    print("귀무가설 채택")
else:
    print("귀무가설 기각")

===일원 분산 분석===
statistic : 34.17427385892114
pvalue : 1.2406415428510513e-10
귀무가설 기각


In [36]:
model = ols('scores~C(groups)',data=df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 scores   R-squared:                       0.740
Model:                            OLS   Adj. R-squared:                  0.718
Method:                 Least Squares   F-statistic:                     34.17
Date:                Sun, 15 Nov 2026   Prob (F-statistic):           1.24e-10
Time:                        15:31:34   Log-Likelihood:                -82.459
No. Observations:                  40   AIC:                             172.9
Df Residuals:                      36   BIC:                             179.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               87.7000      0.634    138.378      0.000      86.415      88.985
C(groups)[T.group_B]    -8.5000      0.896     -9.484      0.000     -10.318      -6.682
C(groups)[T.group_C]    -5.3000      0.896     -5.913      0.000      -7.118      -3.482
C(groups)[T.group_D]    -2.2000      0.896     -2.455      0.019      -4.018      -0.382
==============================================================================
Omnibus:                        2.095   Durbin-Watson:                   2.330
Prob(Omnibus):                  0.351   Jarque-Bera (JB):                1.098
Skew:                          -0.203   Prob(JB):                        0.578
Kurtosis:                       3.703   Cond. No.                         4.79
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [41]:
model_lm = anova_lm(model)
model_lm

,df,sum_sq,mean_sq,F,PR(>F)
C(groups),3.0,411.8,137.266667,34.174274,1.240642e-10
Residual,36.0,144.6,4.016667,NaN,NaN


4.그룹 변수의 자유도를 구하시오.

In [55]:
print("그룹 변수의 자유도 :", model_lm.iloc[0,0])
print("잔차의 자유도 :", model_lm.iloc[1,0])
print("성적의 제곱합 :", model_lm.iloc[0,1])
print("성적의 평균 제곱 :", model_lm.iloc[0,2])
print("f-통계량 값 :", model_lm.iloc[0,3])
print("성적의 p-value :", model_lm.iloc[0,4])

그룹 변수의 자유도 : 3.0
잔차의 자유도 : 36.0
성적의 제곱합 : 411.7999999999971
성적의 평균 제곱 : 137.2666666666657
f-통계량 값 : 34.17427385892093
성적의 p-value : 1.2406415428511555e-10
